In [1]:
import xml.etree.ElementTree as ET


In [2]:

class TOCmanager:
    def __init__(self, file='all_journals_toc.xml'):
        self.file = file
        self.tree = ET.parse(file)
        self.root = self.tree.getroot()
        self.journal2index = {j.attrib['name']: i for i, j in enumerate(self.root)}
        self.journalsList = list(self.journal2index.keys())
        self.journal_tot = [len(j) for j in self.root]
        self.tot = sum(self.journal_tot)
        self.reverse_gaids = dict()
        self.gaids = dict()
        gaid = 0
        for jid, tot in enumerate(self.journal_tot):
            for aid in range(tot):
                self.gaids[gaid] = (jid, aid)
                self.reverse_gaids[(jid, aid)] = gaid
                gaid += 1

    def get(self, jid=0, aid=0):
        if aid >= self.journal_tot[jid]:
            print(f'Journal {self.journalsList[jid]} (JID:{jid}) has {self.journal_tot[jid]} articles. AID requested: {aid}')
            return None
        article = {elem.tag: elem.text for elem in self.root[jid][aid]}
        if len(article.get('Abstract', '') or '') < 100:
            article['Abstract'] = ''
        article['Journal'] = self.journalsList[jid]
        article['jid'] = jid
        article['aid'] = aid
        article['gaid'] = self.reverse_gaids[(jid, aid)]
        return article

    def gaid(self, gaid):
        jid, aid = self.gaids[gaid]
        return self.get(jid, aid)

    def gaid_batch(self, gaids):
        return [self.get(*self.gaids[gaid]) for gaid in gaids]

    def add_field(self, jid, aid, field_name, value):
        """Add or update a field (e.g. keywords) in a specific article."""
        article_elem = self.root[jid][aid]
        existing = article_elem.find(field_name)
        if existing is not None:
            existing.text = value
        else:
            new_elem = ET.SubElement(article_elem, field_name)
            new_elem.text = value

    def save(self, path=None):
        """Write changes back to file."""
        self.tree.write(path or self.file, encoding='utf-8', xml_declaration=True)

    def print(self, article):
        print(f"### Article - JID: {article['jid']} AID: {article['aid']} - GAID: {article['gaid']} ###")
        for k, t in article.items():
            if k in ('aid', 'jid', 'gaid'):
                continue
            print(f"\t{k} : {t}")

    def info(self):
        print("### TOCS ###")
        print(f"\tJournals: {len(self.journal_tot)}")
        print(f"\tArticles: {self.tot}")

    def __str__(self):
        self.info()
        return ''

    def add_field_gaid(self, gaid, field_name, value):
        """Add or update a field in an article given its global article ID (gaid)."""
        if gaid not in self.gaids:
            print(f"Invalid GAID: {gaid}")
            return
        jid, aid = self.gaids[gaid]
        self.add_field(jid, aid, field_name, value)

    def delete_article(self, jid, aid):
        """Delete one article by journal and article index."""
        if jid >= len(self.root):
            print(f"Invalid journal ID: {jid}")
            return
        if aid >= len(self.root[jid]):
            print(f"Journal {self.journalsList[jid]} has only {len(self.root[jid])} articles.")
            return
        # Remove the element
        del self.root[jid][aid]
        # Update bookkeeping
        self.journal_tot[jid] = len(self.root[jid])
        self.tot = sum(self.journal_tot)
        # Rebuild GAID mappings
        self.gaids.clear()
        self.reverse_gaids.clear()
        gaid = 0
        for j, tot in enumerate(self.journal_tot):
            for a in range(tot):
                self.gaids[gaid] = (j, a)
                self.reverse_gaids[(j, a)] = gaid
                gaid += 1
        print(f"Deleted article {aid} from journal {self.journalsList[jid]}")

    def delete_article_gaid(self, gaid):
        """Delete one article using its global article ID (GAID)."""
        if gaid not in self.gaids:
            print(f"Invalid GAID: {gaid}")
            return
        jid, aid = self.gaids[gaid]
        self.delete_article(jid, aid)

    def find_by_doi(self, doi):
        """Return (jid, aid) of an article with the given DOI."""
        for jid, journal in enumerate(self.root):
            for aid, article in enumerate(journal):
                doi_elem = article.find('DOI')
                if doi_elem is not None and doi_elem.text == doi:
                    return jid, aid
        return None

    def delete_by_doi(self, doi):
        """Delete an article by its DOI."""
        res = self.find_by_doi(doi)
        if not res:
            print(f"DOI not found: {doi}")
            return
        jid, aid = res
        self.delete_article(jid, aid)
        print(f"Deleted article with DOI {doi}")

    def add_field_doi(self, doi, field_name, value):
        """Add or update a field in an article identified by its DOI."""
        res = self.find_by_doi(doi)
        if not res:
            print(f"DOI not found: {doi}")
            return
        jid, aid = res
        self.add_field(jid, aid, field_name, value)



import random

tocs = TOCmanager()
tocs.info()
#article = tocs.gaid(10)
#tocs.print(article)
gaids = random.sample(range(tocs.tot), 20)
articles = tocs.gaid_batch(gaids)
prepared = [(article['Title'],article['Abstract']) for article in articles]
prepared

### TOCS ###
	Journals: 47
	Articles: 3771


[('Multimodal prototypical network for interpretable sentiment classification',
  'AbstractRecent advances in sentiment analysis have primarily focused on fusing multimodal information from video data, including visual, acoustic, and textual features, across temporal sequences. While great effort has been made to integrate or fuse information across modalities, less is known about the extent to which temporal segments contribute to model decisions. In addition, current interpretable methods, such as prototype networks, are primarily designed for uni-modal analysis and fail to handle the complex interactions between multiple modalities and temporal dependencies inherent in video data. To address the challenges, we proposeMultiModalPrototypicalNetworks (MMPNet), which extends prototype-based interpretability to multimodal sentiment classification. Specifically, MMPNet can identify contributions of time-level features and leverage them to explain why a particular prediction was made, whil

In [3]:
import requests
import os
from falcon_tests import ArticleClassifierOllama

from dotenv import load_dotenv
load_dotenv()
host = os.getenv('HOST')
ollama = ArticleClassifierOllama(host=host)

In [4]:
for i,art in enumerate(articles):
    print(f'### Article {i} ###')
    out = ollama.classify(art)
    print('\t',art['Title'])
    print('\t',out)


### Article 0 ###
	 Multimodal prototypical network for interpretable sentiment classification
	 {'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['sentiment analysis', 'multimodal information', 'video data', 'prototypical network'], 'specific_keywords': ['temporal sequences', 'interpretable methods', 'visual', 'acoustic', 'textual features']}
### Article 1 ###
	 Antimicrobial treatment ameliorates delirium-like phenotypes in a murine model of urinary tract infection
	 {'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['neurodegeneration', 'microbiome', 'infection', 'behavior'], 'specific_keywords': ['antimicrobials', 'delirium', 'rodent models', 'urinary tract infection']}
### Article 2 ###
	 Altered functional connectivity of reward circuits in adolescents with addictive nonsuicidal self-injury
	 {'neuroscience': 'yes', 'type': 'article', 'generic_keywords': ['addiction', 'adolescence', 'self-injury', 'reward circuits', 'functional connectivity'], 'specific

In [31]:
out = ollama.classify(art)
out

{'neuroscience': 'no',
 'type': 'other',
 'generic_keywords': ['giant panda', 'self-directed tool use', 'behavior'],
 'specific_keywords': ['other']}

In [8]:
ollama.verify_new_keywords()

Amplitude bistability: added (3/3 = 4)
GABAergic neurons: discarded (0/3 = 4)
Gamma band: discarded (0/3 = 4)
Giotto Suite: discarded (0/3 = 4)
GluN2B: added (3/3 = 4)
L-DOPA: discarded (1/3 = 4)
LASSO regression: discarded (0/3 = 4)
Long-range temporal correlations: added (2/3 = 4)
MDD: discarded (0/3 = 4)
NADPH: discarded (0/3 = 4)
NOS isoforms: discarded (0/3 = 4)
NSSI: added (3/3 = 4)
SNARE complexes: added (2/3 = 4)
Theta band: discarded (0/3 = 4)
acoustic: discarded (0/3 = 4)
activity-dependent localization: discarded (1/3 = 4)
aging and exercise: discarded (0/3 = 4)
anterior hippocampal: discarded (1/3 = 4)
antimicrobials: discarded (0/3 = 4)
antipsychotics: discarded (0/3 = 4)
balance: discarded (0/3 = 4)
basal ganglia: discarded (0/3 = 4)
behavioral addictions: discarded (1/3 = 4)
bioinformatics: discarded (0/3 = 4)
brain clearance: discarded (0/3 = 4)
calcium signaling: discarded (0/3 = 4)
center-out reaching task: discarded (0/3 = 4)
cerebral amyloid angiopathy: discarded (1

{'added': ['Amplitude bistability',
  'GluN2B',
  'Long-range temporal correlations',
  'NSSI',
  'SNARE complexes',
  'perceptual narratives',
  'recurrent neural network',
  'substance use disorders'],
 'discarded': ['GABAergic neurons',
  'Gamma band',
  'Giotto Suite',
  'L-DOPA',
  'LASSO regression',
  'MDD',
  'NADPH',
  'NOS isoforms',
  'Theta band',
  'acoustic',
  'activity-dependent localization',
  'aging and exercise',
  'anterior hippocampal',
  'antimicrobials',
  'antipsychotics',
  'balance',
  'basal ganglia',
  'behavioral addictions',
  'bioinformatics',
  'brain clearance',
  'calcium signaling',
  'center-out reaching task',
  'cerebral amyloid angiopathy',
  'computational modeling',
  'conceptual narratives',
  'cytokines',
  'delirium',
  'dopamine',
  'drug effects',
  'dynamics',
  'fMRI',
  'gait',
  'hippocampal connectivity',
  'hippocampus',
  'inflammation',
  'inhibitory signals',
  'interpretable methods',
  'intertemporal choice',
  'kiss-shrink-run 